In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mode pengujian lokal Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan untuk menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Gunakan mode pengujian lokal (tersedia dengan `qiskit-ibm-runtime` v0.22.0 atau lebih baru) untuk menguji program sebelum menyempurnakannya dan mengirimkannya ke perangkat keras kuantum sungguhan. Setelah menggunakan mode pengujian lokal untuk memverifikasi programmu, satu-satunya yang perlu diubah adalah nama Backend untuk menjalankannya di QPU.

Untuk menggunakan mode pengujian lokal, tentukan salah satu fake backend dari ``qiskit_ibm_runtime.fake_provider`` atau tentukan Backend Qiskit Aer saat membuat instance primitif Qiskit Runtime atau sebuah Session.

- **Fake backend**: [Fake backend](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) di ``qiskit_ibm_runtime.fake_provider`` meniru perilaku QPU IBM&reg; dengan menggunakan snapshot QPU. Snapshot QPU berisi informasi penting tentang QPU, seperti coupling map, basis Gate, dan properti Qubit, yang berguna untuk menguji Transpiler dan melakukan simulasi berderau dari QPU. Model noise dari snapshot diterapkan secara otomatis selama simulasi.
- **Simulator Aer**: Simulator dari [Qiskit Aer](/guides/simulate-with-qiskit-aer) memberikan simulasi berkinerja lebih tinggi yang dapat menangani Circuit yang lebih besar dan [model noise kustom](/guides/build-noise-models). Daftar opsi metode simulasi tersedia saat kamu menggunakan `AerSimulator` dalam mode pengujian lokal. Lihat [contoh mode simulasi Clifford](#clifford-sim), yang mendemonstrasikan cara mensimulasikan Circuit Clifford dengan jumlah Qubit yang besar secara efisien.

    <details>
    <summary>**Daftar metode simulasi yang tersedia dari Qiskit Aer**</summary>

    Lihat dokumentasi [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) untuk informasi lebih lanjut.

    * ``"automatic"``: Metode simulasi default. Memilih metode simulasi secara otomatis, berdasarkan Circuit dan model noise.

    * ``"statevector"``: Simulasi statevector padat yang dapat mengambil sampel hasil pengukuran dari Circuit *ideal* dengan semua pengukuran di akhir Circuit. Untuk simulasi berderau, setiap shot mengambil sampel Circuit berderau secara acak dari model noise.

    * ``"density_matrix"``: Simulasi matriks densitas yang dapat mengambil sampel hasil pengukuran dari Circuit *berderau* dengan semua pengukuran di akhir Circuit.

    * ``"stabilizer"``: Simulator state stabilizer Clifford yang efisien yang dapat mensimulasikan Circuit Clifford berderau jika semua error dalam model noise juga merupakan error Clifford.

    * ``"extended_stabilizer"``: Simulator aproksimasi untuk Circuit Clifford + T berdasarkan dekomposisi state menjadi ranked-stabilizer state. Jumlah term bertambah seiring dengan jumlah Gate non-Clifford (T).

    * ``"matrix_product_state"``: Simulator statevector berbasis tensor-network yang menggunakan representasi Matrix Product State (MPS) untuk state. Ini dapat dilakukan dengan atau tanpa memotong dimensi bond MPS, tergantung pada opsi simulator. Perilaku default adalah tanpa pemotongan.

    * ``"unitary"``: Simulasi matriks uniter padat dari Circuit ideal. Ini mensimulasikan matriks uniter dari Circuit itu sendiri, bukan evolusi dari state kuantum awal. Metode ini hanya dapat mensimulasikan Gate; tidak mendukung pengukuran, reset, atau noise.

    * ``"superop"``: Simulasi matriks superoperator padat dari Circuit ideal atau berderau. Ini mensimulasikan matriks superoperator dari Circuit itu sendiri, bukan evolusi dari state kuantum awal. Metode ini dapat mensimulasikan Gate dan reset ideal maupun berderau, tetapi tidak mendukung pengukuran.

    * ``"tensor_network"``: Simulasi berbasis tensor-network yang mendukung statevector dan matriks densitas. Saat ini hanya tersedia untuk GPU dan dipercepat menggunakan API cuQuantum `cuTensorNet`.
    </details>

> **Note:** - Kamu bisa menentukan semua opsi Qiskit Runtime dalam mode pengujian lokal. Namun, semua opsi kecuali shots diabaikan saat dijalankan di simulator lokal.
> - Disarankan untuk menginstal Qiskit Aer sebelum menggunakan fake backend atau simulator Aer dengan menjalankan `pip install qiskit-aer`. Fake backend menggunakan simulator Aer di balik layar jika tersedia, untuk memanfaatkan kinerjanya.


## Contoh fake backend

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Contoh AerSimulator
Contoh dengan Session, tanpa noise:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Untuk mensimulasikan dengan noise, tentukan QPU (perangkat keras kuantum) dan kirimkan ke Aer. Aer membangun model noise berdasarkan data kalibrasi dari QPU tersebut, dan membuat instance Backend Aer dengan model itu. Jika kamu mau, kamu bisa [membangun model noise](/guides/build-noise-models) sendiri.

> **Caution:** QPU dapat dipengaruhi oleh berbagai jenis noise. Model noise Qiskit Aer yang digunakan di sini hanya mensimulasikan sebagian dari noise tersebut dan oleh karena itu kemungkinan besar tidak separah noise pada QPU sungguhan.
> 
> Untuk detail tentang error apa saja yang disertakan saat menginisialisasi model noise dari QPU, lihat referensi API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Contoh dengan noise:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Simulasi Clifford
Karena Circuit Clifford dapat disimulasikan secara efisien dengan hasil yang dapat diverifikasi, simulasi Clifford adalah alat yang sangat berguna. Untuk contoh yang lebih mendalam, lihat [Simulasi efisien Circuit stabilizer dengan primitif Qiskit Aer](/guides/simulate-stabilizer-circuits).

Contoh:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Langkah berikutnya
> **Tip:** - Tinjau [contoh primitif](/guides/primitives-examples) secara detail.
>     - Baca [Migrasi ke primitif V2](/guides/v2-primitives).
>     - Berlatih dengan primitif melalui [pelajaran fungsi biaya](/learning/courses/variational-algorithm-design/cost-functions) di IBM Quantum Learning.
>     - Pelajari cara melakukan transpilasi secara lokal di bagian [Transpile](/guides/transpile).
>     - Coba tutorial [Bandingkan pengaturan Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).